In [ ]:
import csv
import math
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict

"""
Hall & Jones (1999) Replication — Data Merge Pipeline v2
=========================================================
Two versions controlled by a single VERSION switch:

  VERSION = 1  →  Exact replication
                   - PWT 5.6 (1985 prices), cross-section year 1988
                   - ICRG GADP averaged 1986-1995
                   - Barro-Lee 1985 schooling
                   - Sachs-Warner openness fraction 1950-1992

  VERSION = 3  →  Current cross-section
                   - PWT 10.01 (2017 prices), cross-section year 2019
                   - ICRG GADP averaged 2010-2017
                   - Barro-Lee 2015 schooling (updated 2024 dataset)
                   - Fraser Institute Area 5 averaged 2015-2019

Steps that change across versions: 1, 2, 3, 6, 7
Steps that are identical:          4, 5, 8, 9, 10, 11, 12
"""

# =============================================================================
# VERSION SWITCH — change this to 1 or 3
# =============================================================================
VERSION = 1

# =============================================================================
# CONFIG — file paths
# =============================================================================
BASE = "C:\\Users\\Adams\\OneDrive\\DE & E Research\\Data"

# --- Version 1 sources ---
PWT56_FILE  = BASE + "\\GDP, Investment, and Labor force\\pwt56_forweb.xlsx"
BL_V1_FILE  = BASE + "\\Barro-Lee dataset\\BL2013_MF2599_v2.csv"
OPEN_FILE   = BASE + "\\For Index\\SACHS-WARNER OPENNESS INDEX\\open.csv"

# --- Version 3 sources ---
PWT1001_FILE = BASE + "\\GDP, Investment, and Labor force\\pwt1001.xlsx"
BL_V3_FILE   = BASE + "\\Barro-Lee dataset\\OUP_proj_MF2564_v1.csv"
FRASER_FILE  = BASE + "\\For Index\\SACHS-WARNER OPENNESS INDEX\\efotw-2025-master-index-data-for-researchers-iso.xlsx"

# --- Shared sources (same for both versions) ---
ICRG_FILE    = BASE + "\\For Index\\World Bank Governance Indicators\\3BResearchersDataset2017.xlsx"
MINING_FILE  = BASE + "\\MINING VALUE ADDED\\Contribution of mining to value added.xlsx"
GEO_FILE     = BASE + "\\INSTRUMENTAL Variables Data\\geo_instruments.csv"
FR_FILE      = BASE + "\\INSTRUMENTAL Variables Data\\frankel_romer_trade.csv"
LANG_FILE    = BASE + "\\INSTRUMENTAL Variables Data\\language_instruments.csv"

# --- Output file ---
OUTPUT_FILE  = f"merged_v{VERSION}.csv"

# --- Version-specific settings ---
if VERSION == 1:
    CROSS_SECTION_YEAR  = 1988
    BL_YEAR             = '1985'
    ICRG_GOV_YEARS      = list(range(1986, 1996))   # 1986-1995 inclusive
    PWT_FILE            = PWT56_FILE
    BL_FILE             = BL_V1_FILE
    MINING_PROXY_YEAR   = '1990'
    print("VERSION 1: Exact replication — PWT 5.6, ICRG 1986-1995, BL 1985, Sachs-Warner")

elif VERSION == 3:
    CROSS_SECTION_YEAR  = 2019
    BL_YEAR             = '2015'
    ICRG_GOV_YEARS      = list(range(2010, 2018))   # 2010-2017 inclusive
    PWT_FILE            = PWT1001_FILE
    BL_FILE             = BL_V3_FILE
    MINING_PROXY_YEAR   = '2018'                    # most recent available
    print("VERSION 3: Current cross-section — PWT 10.01, ICRG 2010-2017, BL 2015, Fraser 2015-2019")

else:
    raise ValueError("VERSION must be 1 or 3")

print(f"  Cross-section year : {CROSS_SECTION_YEAR}")
print(f"  Education year     : {BL_YEAR}")
print(f"  Governance window  : {ICRG_GOV_YEARS[0]}-{ICRG_GOV_YEARS[-1]}")
print(f"  Output file        : {OUTPUT_FILE}")


In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def read_xlsx_sheet(filepath, sheet_index=0):
    """
    Read a sheet from an xlsx file.
    Returns (headers, rows) where rows is a list of dicts.
    sheet_index: 0-based index of the sheet to read.
    """
    with zipfile.ZipFile(filepath, 'r') as z:
        with z.open('xl/sharedStrings.xml') as f:
            tree = ET.parse(f)
            root = tree.getroot()
            ns = {'ns': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            strings = []
            for si in root.findall('ns:si', ns):
                t = si.find('ns:t', ns)
                if t is not None:
                    strings.append(t.text or '')
                else:
                    texts = si.findall('.//ns:t', ns)
                    strings.append(''.join(t.text or '' for t in texts))

        sheet_files = sorted([
            n for n in z.namelist()
            if 'worksheets/sheet' in n and n.endswith('.xml')
        ])

        def get_val(cell, strings):
            t = cell.get('t')
            v = cell.find('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v')
            if v is None:
                return None
            if t == 's':
                idx = int(v.text)
                return strings[idx] if idx < len(strings) else None
            return v.text

        with z.open(sheet_files[sheet_index]) as f:
            tree = ET.parse(f)
            root = tree.getroot()
            rows = root.findall(
                './/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row'
            )

        if not rows:
            return [], []

        headers = [get_val(c, strings) for c in rows[0]]
        data = []
        for row in rows[1:]:
            vals = [get_val(c, strings) for c in row]
            d = dict(zip(headers, vals))
            data.append(d)

        return headers, data


def read_xls_binary(filepath):
    """
    Read a legacy .xls (BIFF8) file using the xlrd library if available,
    otherwise attempt to parse binary content.
    Returns (headers, rows) where rows is a list of dicts.
    Used for ICRG 3BResearchersDataset2017.xls
    """
    try:
        import xlrd
        wb = xlrd.open_workbook(filepath)
        ws = wb.sheets()[0]
        headers = [str(ws.cell_value(0, c)).strip() for c in range(ws.ncols)]
        data = []
        for r in range(1, ws.nrows):
            row = {}
            for c, h in enumerate(headers):
                val = ws.cell_value(r, c)
                row[h] = val if val != '' else None
            data.append(row)
        return headers, data
    except ImportError:
        raise ImportError(
            "The xlrd library is required to read .xls files.\n"
            "Install it with: pip install xlrd"
        )


def safe_float(val, default=None):
    """Convert a value to float, returning default if conversion fails."""
    if val is None or val == '':
        return default
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


In [ ]:
# =============================================================================
# COUNTRY NAME CROSSWALKS
# =============================================================================
# PWT 5.6 country names — exact spellings found in pwt56_forweb.xlsx
PWT56_NAME_TO_ISO = {
    'ALGERIA': 'DZA', 'ANGOLA': 'AGO', 'BENIN': 'BEN', 'BOTSWANA': 'BWA',
    'BURKINA FASO': 'BFA', 'BURUNDI': 'BDI', 'CAMEROON': 'CMR',
    'CAPE VERDE IS.': 'CPV', 'CENTRAL AFR.R.': 'CAF', 'CHAD': 'TCD',
    'COMOROS': 'COM', 'CONGO': 'COG', 'EGYPT': 'EGY', 'ETHIOPIA': 'ETH',
    'GABON': 'GAB', 'GAMBIA': 'GMB', 'GHANA': 'GHA', 'GUINEA': 'GIN',
    'GUINEA-BISS': 'GNB', 'IVORY COAST': 'CIV',
    'KENYA': 'KEN', 'LESOTHO': 'LSO', 'LIBERIA': 'LBR',
    'MADAGASCAR': 'MDG', 'MALAWI': 'MWI', 'MALI': 'MLI',
    'MAURITANIA': 'MRT', 'MAURITIUS': 'MUS', 'MOROCCO': 'MAR',
    'MOZAMBIQUE': 'MOZ', 'NAMIBIA': 'NAM', 'NIGER': 'NER',
    'NIGERIA': 'NGA', 'REUNION': 'REU', 'RWANDA': 'RWA', 'SENEGAL': 'SEN',
    'SEYCHELLES': 'SYC', 'SIERRA LEONE': 'SLE', 'SOMALIA': 'SOM',
    'SOUTH AFRICA': 'ZAF', 'SUDAN': 'SDN', 'SWAZILAND': 'SWZ',
    'TANZANIA': 'TZA', 'TOGO': 'TGO', 'TUNISIA': 'TUN', 'UGANDA': 'UGA',
    'ZAIRE': 'ZAR', 'ZAMBIA': 'ZMB', 'ZIMBABWE': 'ZWE',
    'DJIBOUTI': 'DJI',
    'BAHAMAS': 'BHS', 'BARBADOS': 'BRB', 'BELIZE': 'BLZ', 'CANADA': 'CAN',
    'COSTA RICA': 'CRI', 'DOMINICA': 'DMA', 'DOMINICAN REP.': 'DOM',
    'EL SALVADOR': 'SLV', 'GRENADA': 'GRD', 'GUATEMALA': 'GTM',
    'HAITI': 'HTI', 'HONDURAS': 'HND', 'JAMAICA': 'JAM', 'MEXICO': 'MEX',
    'NICARAGUA': 'NIC', 'PANAMA': 'PAN', 'PUERTO RICO': 'PRI',
    'TRINIDAD&TOBAGO': 'TTO', 'U.S.A.': 'USA',
    'ST.KITTS&NEVIS': 'KNA', 'ST.LUCIA': 'LCA', 'ST.VINCENT&GRE': 'VCT',
    'ARGENTINA': 'ARG', 'BOLIVIA': 'BOL', 'BRAZIL': 'BRA', 'CHILE': 'CHL',
    'COLOMBIA': 'COL', 'ECUADOR': 'ECU', 'GUYANA': 'GUY',
    'PARAGUAY': 'PRY', 'PERU': 'PER', 'SURINAME': 'SUR', 'URUGUAY': 'URY',
    'VENEZUELA': 'VEN',
    'BAHRAIN': 'BHR', 'BANGLADESH': 'BGD', 'BHUTAN': 'BTN',
    'CHINA': 'CHN', 'HONG KONG': 'HKG', 'INDIA': 'IND',
    'INDONESIA': 'IDN', 'IRAN': 'IRN', 'IRAQ': 'IRQ', 'ISRAEL': 'ISR',
    'JAPAN': 'JPN', 'JORDAN': 'JOR', 'KOREA, REP.': 'KOR',
    'KUWAIT': 'KWT', 'LAOS': 'LAO', 'MALAYSIA': 'MYS', 'MONGOLIA': 'MNG',
    'MYANMAR': 'MMR', 'NEPAL': 'NPL', 'OMAN': 'OMN', 'PAKISTAN': 'PAK',
    'PHILIPPINES': 'PHL', 'QATAR': 'QAT', 'SAUDI ARABIA': 'SAU',
    'SINGAPORE': 'SGP', 'SRI LANKA': 'LKA', 'SYRIA': 'SYR',
    'TAIWAN': 'TWN', 'THAILAND': 'THA', 'UNITED ARAB E.': 'ARE',
    'YEMEN': 'YEM',
    'AUSTRIA': 'AUT', 'BELGIUM': 'BEL', 'BULGARIA': 'BGR', 'CYPRUS': 'CYP',
    'CZECHOSLOVAKIA': 'CZE', 'DENMARK': 'DNK', 'FINLAND': 'FIN',
    'FRANCE': 'FRA', 'GERMANY, WEST': 'DEU', 'GERMANY, EAST': 'DDR',
    'GREECE': 'GRC', 'HUNGARY': 'HUN', 'ICELAND': 'ISL', 'IRELAND': 'IRL',
    'ITALY': 'ITA', 'LUXEMBOURG': 'LUX', 'MALTA': 'MLT',
    'NETHERLANDS': 'NLD', 'NORWAY': 'NOR', 'POLAND': 'POL',
    'PORTUGAL': 'PRT', 'ROMANIA': 'ROU', 'SPAIN': 'ESP', 'SWEDEN': 'SWE',
    'SWITZERLAND': 'CHE', 'TURKEY': 'TUR', 'U.K.': 'GBR',
    'U.S.S.R.': 'SUN', 'YUGOSLAVIA': 'YUG',
    'AUSTRALIA': 'AUS', 'FIJI': 'FJI', 'NEW ZEALAND': 'NZL',
    'PAPUA N.GUINEA': 'PNG', 'SOLOMON IS.': 'SLB', 'VANUATU': 'VUT',
    'TONGA': 'TON', 'WESTERN SAMOA': 'WSM',
}

# ICRG country names
ICRG_NAME_TO_ISO = {
    'Albania': 'ALB', 'Algeria': 'DZA', 'Angola': 'AGO', 'Argentina': 'ARG',
    'Armenia': 'ARM', 'Australia': 'AUS', 'Austria': 'AUT',
    'Azerbaijan': 'AZE', 'Bahamas': 'BHS', 'Bahrain': 'BHR',
    'Bangladesh': 'BGD', 'Belarus': 'BLR', 'Belgium': 'BEL',
    'Bolivia': 'BOL', 'Botswana': 'BWA', 'Brazil': 'BRA', 'Brunei': 'BRN',
    'Bulgaria': 'BGR', 'Burkina Faso': 'BFA', 'Cameroon': 'CMR',
    'Canada': 'CAN', 'Chile': 'CHL', 'China': 'CHN', 'Colombia': 'COL',
    'Congo': 'COG', 'Congo, DR': 'COD', "Cote d'Ivoire": 'CIV',
    'Croatia': 'HRV', 'Cuba': 'CUB', 'Cyprus': 'CYP',
    'Czech Republic': 'CZE', 'Czechoslovakia': 'CZE', 'Denmark': 'DNK',
    'Dominican Republic': 'DOM', 'East Germany': 'DDR', 'Ecuador': 'ECU',
    'Egypt': 'EGY', 'El Salvador': 'SLV', 'Estonia': 'EST',
    'Ethiopia': 'ETH', 'Finland': 'FIN', 'France': 'FRA', 'Gabon': 'GAB',
    'Gambia': 'GMB', 'Germany': 'DEU', 'Ghana': 'GHA', 'Greece': 'GRC',
    'Guatemala': 'GTM', 'Guinea': 'GIN', 'Guinea-Bissau': 'GNB',
    'Guyana': 'GUY', 'Haiti': 'HTI', 'Honduras': 'HND',
    'Hong Kong': 'HKG', 'Hungary': 'HUN', 'Iceland': 'ISL', 'India': 'IND',
    'Indonesia': 'IDN', 'Iran': 'IRN', 'Iraq': 'IRQ', 'Ireland': 'IRL',
    'Israel': 'ISR', 'Italy': 'ITA', 'Jamaica': 'JAM', 'Japan': 'JPN',
    'Jordan': 'JOR', 'Kenya': 'KEN', 'Korea, DPR': 'PRK',
    'Kuwait': 'KWT', 'Latvia': 'LVA', 'Lebanon': 'LBN', 'Liberia': 'LBR',
    'Libya': 'LBY', 'Lithuania': 'LTU', 'Luxembourg': 'LUX',
    'Madagascar': 'MDG', 'Malawi': 'MWI', 'Malaysia': 'MYS', 'Mali': 'MLI',
    'Malta': 'MLT', 'Mexico': 'MEX', 'Moldova': 'MDA', 'Mongolia': 'MNG',
    'Morocco': 'MAR', 'Mozambique': 'MOZ', 'Myanmar': 'MMR',
    'Namibia': 'NAM', 'Netherlands': 'NLD', 'New Zealand': 'NZL',
    'Nicaragua': 'NIC', 'Niger': 'NER', 'Nigeria': 'NGA', 'Norway': 'NOR',
    'Oman': 'OMN', 'Pakistan': 'PAK', 'Panama': 'PAN',
    'Papua New Guinea': 'PNG', 'Paraguay': 'PRY', 'Peru': 'PER',
    'Philippines': 'PHL', 'Poland': 'POL', 'Portugal': 'PRT',
    'Qatar': 'QAT', 'Romania': 'ROU', 'Russia': 'RUS',
    'Saudi Arabia': 'SAU', 'Senegal': 'SEN', 'Sierra Leone': 'SLE',
    'Singapore': 'SGP', 'Slovakia': 'SVK', 'Slovenia': 'SVN',
    'Somalia': 'SOM', 'South Africa': 'ZAF', 'South Korea': 'KOR',
    'Spain': 'ESP', 'Sri Lanka': 'LKA', 'Sudan': 'SDN',
    'Suriname': 'SUR', 'Sweden': 'SWE', 'Switzerland': 'CHE',
    'Syria': 'SYR', 'Taiwan': 'TWN', 'Tanzania': 'TZA', 'Thailand': 'THA',
    'Togo': 'TGO', 'Trinidad & Tobago': 'TTO', 'Tunisia': 'TUN',
    'Turkey': 'TUR', 'Uganda': 'UGA', 'Ukraine': 'UKR',
    'United Kingdom': 'GBR', 'United States': 'USA', 'Uruguay': 'URY',
    'USSR': 'SUN', 'Venezuela': 'VEN', 'Vietnam': 'VNM',
    'West Germany': 'DEU', 'Yemen': 'YEM', 'Zambia': 'ZMB',
    'Zimbabwe': 'ZWE', 'Serbia': 'SRB', 'Serbia & Montenegro': 'SCG',
    'Kazakhstan': 'KAZ', 'KazakhstanU': 'KAZ',
    'Burundi': 'BDI', 'Benin': 'BEN', 'Belize': 'BLZ',
    'Barbados': 'BRB', 'Central African Republic': 'CAF',
    'Comoros': 'COM', 'Costa Rica': 'CRI', 'Fiji': 'FJI',
    'Lesotho': 'LSO', 'Mauritania': 'MRT', 'Mauritius': 'MUS',
    'Puerto Rico': 'PRI', 'Rwanda': 'RWA', 'Swaziland': 'SWZ',
    'Seychelles': 'SYC', 'Chad': 'TCD', 'Trinidad and Tobago': 'TTO',
    'Vanuatu': 'VUT', 'Yugoslavia': 'YUG', 'Zaire': 'ZAR',
}

print("Crosswalks loaded.")
print(f"  PWT 5.6 entries : {len(PWT56_NAME_TO_ISO)}")
print(f"  ICRG entries    : {len(ICRG_NAME_TO_ISO)}")


In [ ]:
# =============================================================================
# STEP 1: Output per worker — branches on VERSION
# =============================================================================
print(f"\n>>> STEP 1: Loading output per worker (VERSION {VERSION})...")

master = {}

if VERSION == 1:
    _, pwt_rows = read_xlsx_sheet(PWT_FILE, sheet_index=0)
    print(f"  PWT 5.6 total rows: {len(pwt_rows)}")

    for row in pwt_rows:
        country_name = (row.get('Country') or '').strip()
        iso = PWT56_NAME_TO_ISO.get(country_name)
        if not iso:
            continue

        yr = str(row.get('Year') or '').strip()
        if yr != str(CROSS_SECTION_YEAR):
            continue

        rgdpw = safe_float(row.get('RGDPW'))
        if rgdpw is None or rgdpw <= 0:
            continue

        kapw = safe_float(row.get('KAPW'))

        # Component columns are percentage SHARES of KAPW
        # Non-residential = KDUR + KNRES + KOTHER + KTRANP (excludes KRES)
        kdur   = safe_float(row.get('KDUR'))
        knres  = safe_float(row.get('KNRES'))
        kother = safe_float(row.get('KOTHER'))
        kres   = safe_float(row.get('KRES'))
        ktranp = safe_float(row.get('KTRANP'))

        # Compute non-residential K:
        #   If all non-res components available: K_nonres = KAPW * (sum_nonres/sum_total)
        #   where sum_nonres = KDUR+KNRES+KOTHER+KTRANP, sum_total adds KRES
        #   This removes residential share from total capital stock
        # If components missing: fall back to total KAPW (flag as approximate)
        nonres_components = [x for x in [kdur, knres, kother, ktranp] if x is not None]
        all_components    = [x for x in [kdur, knres, kother, kres, ktranp] if x is not None]

        if kapw and len(nonres_components) == 4 and len(all_components) == 5:
            # Full decomposition available
            sum_nonres = sum(nonres_components)
            sum_total  = sum(all_components)
            k_nonres = kapw * (sum_nonres / sum_total)
            kapw_flag = 'full'
        elif kapw and len(nonres_components) >= 2:
            # Partial decomposition — use ratio of available nonres to available total
            sum_nonres = sum(nonres_components)
            sum_total  = sum(all_components) if all_components else sum_nonres
            k_nonres = kapw * (sum_nonres / sum_total)
            kapw_flag = 'partial'
        elif kapw:
            # No component breakdown — use total KAPW as approximation
            k_nonres = kapw
            kapw_flag = 'total_only'
        else:
            k_nonres = None
            kapw_flag = 'missing'

        master[iso] = {
            'iso3':        iso,
            'country':     country_name,
            'yl':          rgdpw,
            'kapw_total':  kapw,
            'kapw_nonres': k_nonres,
            'kapw_flag':   kapw_flag,
        }

    print(f"  Countries loaded (year={CROSS_SECTION_YEAR}): {len(master)}")
    flags = {}
    for d in master.values():
        f = d.get('kapw_flag','?')
        flags[f] = flags.get(f,0)+1
    for f, n in sorted(flags.items()):
        print(f"    KAPW decomposition '{f}': {n} countries")

elif VERSION == 3:
    _, pwt_rows = read_xlsx_sheet(PWT_FILE, sheet_index=2)
    print(f"  PWT 10.01 total rows: {len(pwt_rows)}")

    all_pwt = defaultdict(dict)
    pwt_meta = {}

    for row in pwt_rows:
        iso    = (row.get('countrycode') or '').strip()
        yr_str = (row.get('year') or '').strip()
        if not iso or not yr_str:
            continue
        try:
            yr = int(float(yr_str))
        except:
            continue

        rgdpo = safe_float(row.get('rgdpo'))
        emp   = safe_float(row.get('emp'))
        csh_i = safe_float(row.get('csh_i'))
        inv   = rgdpo * csh_i if (rgdpo and csh_i) else None
        all_pwt[iso][yr] = inv

        if yr == CROSS_SECTION_YEAR:
            yl = (rgdpo / emp) if (rgdpo and emp and emp > 0) else None
            pwt_meta[iso] = {
                'iso3':    iso,
                'country': row.get('country', ''),
                'yl':      yl,
                'rgdpo':   rgdpo,
                'emp':     emp,
                'csh_i':   csh_i,
                'delta':   safe_float(row.get('delta')),
                'labsh':   safe_float(row.get('labsh')),
            }

    for iso, d in pwt_meta.items():
        if d.get('yl') is not None:
            master[iso] = dict(d)

    print(f"  Countries with Y/L data (year={CROSS_SECTION_YEAR}): {len(master)}")

print(f"\n  Sample Y/L values:")
for check in ['USA', 'NER', 'DEU', 'JPN', 'GBR', 'CHN', 'KEN']:
    yl = master.get(check, {}).get('yl')
    print(f"    {check}: {yl:,.0f}" if yl else f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 2: Capital stock — branches on VERSION
# =============================================================================
print(f"\n>>> STEP 2: Capital stock (VERSION {VERSION})...")

DELTA = 0.06

if VERSION == 1:
    # K/Y = K_nonres_per_worker / RGDPW
    # Both from PWT 5.6, same price base (1985 intl $)
    matched = 0
    for iso, d in master.items():
        kapw = d.get('kapw_nonres')
        yl   = d.get('yl')
        if kapw is not None and yl and yl > 0:
            d['K']        = kapw
            d['ky_ratio'] = kapw / yl
            matched += 1
        else:
            d['K']        = None
            d['ky_ratio'] = None

    print(f"  K/Y computed for: {matched} countries")
    print(f"  Method: KAPW × (non-res share) / RGDPW")
    print(f"  Non-res components: KDUR + KNRES + KOTHER + KTRANP (excludes KRES)")

elif VERSION == 3:
    capital_stocks = {}
    for iso, inv_series in all_pwt.items():
        years_with_data = sorted(y for y, v in inv_series.items() if v is not None)
        if not years_with_data:
            continue
        if CROSS_SECTION_YEAR not in years_with_data and max(years_with_data) < CROSS_SECTION_YEAR:
            continue

        first_year  = years_with_data[0]
        early_years = [y for y in years_with_data if y <= first_year + 10]
        if len(early_years) >= 2:
            i_start = inv_series[early_years[0]]
            i_end   = inv_series[early_years[-1]]
            n_yrs   = early_years[-1] - early_years[0]
            if i_start and i_end and i_start > 0 and i_end > 0 and n_yrs > 0:
                g = (i_end / i_start) ** (1 / n_yrs) - 1
                g = max(0.01, min(g, 0.10))
            else:
                g = 0.02
        else:
            g = 0.02

        i0 = inv_series.get(first_year)
        K  = (i0 / (g + DELTA)) if i0 else None

        for yr in range(first_year + 1, CROSS_SECTION_YEAR + 1):
            inv = inv_series.get(yr)
            if K is not None and inv is not None:
                K = inv + (1 - DELTA) * K
            elif K is not None:
                K = (1 - DELTA) * K

        if K is not None and K > 0:
            capital_stocks[iso] = K

    matched = 0
    for iso, d in master.items():
        K   = capital_stocks.get(iso)
        yl  = d.get('yl')
        emp = d.get('emp')
        if K is not None and yl and emp and emp > 0:
            kpw = K / emp
            d['K']        = kpw
            d['ky_ratio'] = kpw / yl
            matched += 1
        else:
            d['K']        = None
            d['ky_ratio'] = None

    print(f"  K/Y computed for: {matched} countries (perpetual inventory, delta={DELTA})")

# Sanity check
print(f"\n  Sample K/Y ratios:")
usa_ky = master.get('USA', {}).get('ky_ratio', 1)
for check in ['USA', 'DEU', 'JPN', 'GBR', 'BRA', 'CHN', 'KEN', 'NER']:
    ky = master.get(check, {}).get('ky_ratio')
    print(f"    {check}: K/Y = {ky:.4f}" if ky else f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 3: Barro-Lee Education
# =============================================================================
print(f"\n>>> STEP 3: Loading Barro-Lee education data (year={BL_YEAR})...")

# Both V1 (BL2013_MF2599_v2.csv) and V3 (OUP_proj_MF2564_v1.csv) share:
#   - WBcode column for ISO identifier
#   - yr_sch column for average years of schooling
#   - year column for the 5-year interval
#   - sex == 'MF' for both sexes combined
#   - agefrom == '25' for 25+ (V1) or 25-64 (V3)

bl_data = {}
with open(BL_FILE, 'r', encoding='latin1') as f:
    reader = csv.DictReader(f)
    for row in reader:
        yr  = str(row.get('year', '')).strip()
        sex = str(row.get('sex', '')).strip()
        age = str(row.get('agefrom', '')).strip()
        if yr != BL_YEAR or sex != 'MF' or age != '25':
            continue
        iso = (row.get('WBcode') or row.get('wbcode') or '').strip()
        yr_sch = safe_float(row.get('yr_sch'))
        if iso:
            bl_data[iso] = yr_sch

print(f"  Countries with {BL_YEAR} education data: {len(bl_data)}")

if VERSION == 3:
    print(f"  Note: V3 uses age 25-64 (not 25+). Minor deviation documented.")

# =============================================================================
# STEP 4: Human Capital Index (piecewise Mincerian returns)
# =============================================================================
print(f"\n>>> STEP 4: Computing human capital index...")

# phi(E) = 0.134*min(E,4) + 0.101*max(min(E,8)-4,0) + 0.068*max(E-8,0)
# h = exp(phi(E))
# Returns: 13.4% for years 1-4, 10.1% for years 5-8, 6.8% for years 9+
# Source: Psacharopoulos (1994)

def mincerian_phi(E):
    if E is None:
        return None
    e1 = min(E, 4)
    e2 = max(min(E, 8) - 4, 0)
    e3 = max(E - 8, 0)
    return 0.134 * e1 + 0.101 * e2 + 0.068 * e3

# Merge education and human capital into master
for iso, d in master.items():
    yr_sch = bl_data.get(iso)
    d['yr_sch'] = yr_sch
    phi = mincerian_phi(yr_sch)
    d['hc_bl'] = math.exp(phi) if phi is not None else None

# Use hc_bl as primary human capital measure (hc column)
for iso, d in master.items():
    d['hc'] = d.get('hc_bl')

n_hc = sum(1 for d in master.values() if d.get('hc') is not None)
print(f"  Human capital computed for: {n_hc} countries")

# Sanity checks
print(f"\n  Sample human capital values:")
for check in ['USA', 'NER', 'JPN', 'KEN', 'DEU']:
    sch = master.get(check, {}).get('yr_sch')
    hc  = master.get(check, {}).get('hc')
    if hc:
        print(f"    {check}: yr_sch={sch:.2f}, h={hc:.4f}" if sch else f"    {check}: h={hc:.4f}")
    else:
        print(f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 5: Mining Value Added
# =============================================================================
print(f"\n>>> STEP 5: Loading mining value added...")

# Mining file uses country names — apply NAME_TO_ISO crosswalk
# VERSION 1: use 1990 as proxy for 1988
# VERSION 3: use most recent available year (MINING_PROXY_YEAR)

MINING_NAME_TO_ISO = {
    'Afghanistan': 'AFG', 'Albania': 'ALB', 'Algeria': 'DZA', 'Angola': 'AGO',
    'Argentina': 'ARG', 'Armenia': 'ARM', 'Australia': 'AUS', 'Austria': 'AUT',
    'Azerbaijan': 'AZE', 'Bahrain': 'BHR', 'Bangladesh': 'BGD', 'Belarus': 'BLR',
    'Belgium': 'BEL', 'Belize': 'BLZ', 'Benin': 'BEN', 'Bolivia': 'BOL',
    'Bosnia and Herzegovina': 'BIH', 'Botswana': 'BWA', 'Brazil': 'BRA',
    'Brunei Darussalam': 'BRN', 'Bulgaria': 'BGR', 'Burkina Faso': 'BFA',
    'Burundi': 'BDI', 'Cabo Verde': 'CPV', 'Cambodia': 'KHM', 'Cameroon': 'CMR',
    'Canada': 'CAN', 'Central African Republic': 'CAF', 'Chad': 'TCD',
    'Chile': 'CHL', 'China': 'CHN', 'Colombia': 'COL', 'Comoros': 'COM',
    'Congo': 'COG', 'Costa Rica': 'CRI', 'Croatia': 'HRV', 'Cuba': 'CUB',
    'Cyprus': 'CYP', 'Czech Republic': 'CZE', 'Czechia': 'CZE',
    'Democratic Republic of the Congo': 'COD', 'Denmark': 'DNK',
    'Djibouti': 'DJI', 'Dominican Republic': 'DOM', 'Ecuador': 'ECU',
    'Egypt': 'EGY', 'El Salvador': 'SLV', 'Equatorial Guinea': 'GNQ',
    'Eritrea': 'ERI', 'Estonia': 'EST', 'Eswatini': 'SWZ', 'Ethiopia': 'ETH',
    'Fiji': 'FJI', 'Finland': 'FIN', 'France': 'FRA', 'Gabon': 'GAB',
    'Gambia': 'GMB', 'Georgia': 'GEO', 'Germany': 'DEU', 'Ghana': 'GHA',
    'Greece': 'GRC', 'Guatemala': 'GTM', 'Guinea': 'GIN',
    'Guinea-Bissau': 'GNB', 'Guyana': 'GUY', 'Haiti': 'HTI',
    'Honduras': 'HND', 'Hungary': 'HUN', 'Iceland': 'ISL', 'India': 'IND',
    'Indonesia': 'IDN', 'Iran': 'IRN', 'Iran (Islamic Republic of)': 'IRN',
    'Iraq': 'IRQ', 'Ireland': 'IRL', 'Israel': 'ISR', 'Italy': 'ITA',
    'Jamaica': 'JAM', 'Japan': 'JPN', 'Jordan': 'JOR', 'Kazakhstan': 'KAZ',
    'Kenya': 'KEN', 'Kuwait': 'KWT', 'Kyrgyzstan': 'KGZ', 'Lao PDR': 'LAO',
    'Latvia': 'LVA', 'Lebanon': 'LBN', 'Lesotho': 'LSO', 'Liberia': 'LBR',
    'Libya': 'LBY', 'Lithuania': 'LTU', 'Luxembourg': 'LUX',
    'Madagascar': 'MDG', 'Malawi': 'MWI', 'Malaysia': 'MYS', 'Mali': 'MLI',
    'Malta': 'MLT', 'Mauritania': 'MRT', 'Mauritius': 'MUS', 'Mexico': 'MEX',
    'Moldova': 'MDA', 'Mongolia': 'MNG', 'Morocco': 'MAR', 'Mozambique': 'MOZ',
    'Myanmar': 'MMR', 'Namibia': 'NAM', 'Nepal': 'NPL', 'Netherlands': 'NLD',
    'New Zealand': 'NZL', 'Nicaragua': 'NIC', 'Niger': 'NER', 'Nigeria': 'NGA',
    'Norway': 'NOR', 'Oman': 'OMN', 'Pakistan': 'PAK', 'Panama': 'PAN',
    'Papua New Guinea': 'PNG', 'Paraguay': 'PRY', 'Peru': 'PER',
    'Philippines': 'PHL', 'Poland': 'POL', 'Portugal': 'PRT', 'Qatar': 'QAT',
    'Romania': 'ROU', 'Russia': 'RUS', 'Russian Federation': 'RUS',
    'Rwanda': 'RWA', 'Saudi Arabia': 'SAU', 'Senegal': 'SEN',
    'Sierra Leone': 'SLE', 'Singapore': 'SGP', 'Slovakia': 'SVK',
    'Slovenia': 'SVN', 'Somalia': 'SOM', 'South Africa': 'ZAF',
    'South Korea': 'KOR', 'Spain': 'ESP', 'Sri Lanka': 'LKA', 'Sudan': 'SDN',
    'Suriname': 'SUR', 'Sweden': 'SWE', 'Switzerland': 'CHE', 'Syria': 'SYR',
    'Syrian Arab Republic': 'SYR', 'Taiwan': 'TWN', 'Tanzania': 'TZA',
    'Thailand': 'THA', 'Togo': 'TGO', 'Trinidad and Tobago': 'TTO',
    'Tunisia': 'TUN', 'Turkey': 'TUR', 'Uganda': 'UGA', 'Ukraine': 'UKR',
    'United Arab Emirates': 'ARE', 'United Kingdom': 'GBR',
    'United States': 'USA', 'Uruguay': 'URY', 'Uzbekistan': 'UZB',
    'Venezuela': 'VEN', 'Viet Nam': 'VNM', 'Vietnam': 'VNM',
    'Yemen': 'YEM', 'Zambia': 'ZMB', 'Zimbabwe': 'ZWE',
    "Côte d'Ivoire": 'CIV', "Cote d'Ivoire": 'CIV', 'Ivory Coast': 'CIV',
    'Congo, Dem. Rep.': 'COD', 'Congo, Rep.': 'COG',
    'Republic of Korea': 'KOR', 'Korea, Republic of': 'KOR',
    'Swaziland': 'SWZ', 'Cape Verde': 'CPV', 'Bahamas': 'BHS',
    'Barbados': 'BRB', 'Bhutan': 'BTN', 'Comoros': 'COM',
    'Bolivia (Plurinational State of)': 'BOL',
}

mining_data = {}
_, mining_rows = read_xlsx_sheet(MINING_FILE, sheet_index=1)

for row in mining_rows:
    country_name = (row.get('Country and area') or '').strip()
    if not country_name:
        continue
    iso = MINING_NAME_TO_ISO.get(country_name)
    if not iso:
        continue
    val = safe_float(row.get(MINING_PROXY_YEAR))
    # Fallback to nearby years if primary not available
    if val is None:
        for fallback_yr in ['2017', '2016', '2015', '1995', '1992', '1990']:
            val = safe_float(row.get(fallback_yr))
            if val is not None:
                break
    if val is not None:
        mining_data[iso] = val / 100   # % to fraction

print(f"  Mining data loaded for: {len(mining_data)} countries")
print(f"  Using year: {MINING_PROXY_YEAR} (with fallback to nearby years)")

for iso, d in master.items():
    d['mining_va'] = mining_data.get(iso, 0.0)   # default 0 if missing

matched_mining = sum(1 for v in master.values() if v.get('mining_va') is not None)
print(f"  Matched: {matched_mining} countries")

# Sample checks
for check in ['USA', 'NER', 'NGA', 'SAU', 'NOR']:
    mv = master.get(check, {}).get('mining_va')
    print(f"    {check} mining share: {mv:.3f}" if mv is not None else f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 6: ICRG GADP Governance Index
# =============================================================================
import pandas as pd
import re
from functools import reduce
import math

print(f"\n>>> STEP 6: Constructing ICRG GADP (avg {ICRG_GOV_YEARS[0]}-{ICRG_GOV_YEARS[-1]})...")

ENGINE = "openpyxl"

# Five GADP components: sheet name prefix -> (dict key, max score)
GADP_SHEETS = {
    "A-Government Stability":      ("A-GovStab",  12),
    "F-Corruption":                ("F-Corrupt",   6),
    "I-Law and Order":             ("I-Law",        6),
    "K-Democratic Accountability": ("K-DemoAcct",  6),
    "L-Bureaucracy Quality":       ("L-BureauY",   4),
}

def _is_year_token(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return False
    try:
        fv = float(str(val).strip().rstrip(".0") + ("0" if str(val).strip().endswith(".") else ""))
        fv = float(val)
        if abs(fv - round(fv)) < 1e-9:
            y = int(round(fv))
            return 1800 <= y <= 2100
    except Exception:
        pass
    return False

def _find_header_row(df, min_years=5):
    """Find the row containing 'Country' and year columns."""
    for r in range(min(300, len(df))):
        row = df.iloc[r, :].tolist()
        has_country = any(str(v).strip().lower() == "country" for v in row if pd.notna(v))
        n_years = sum(1 for v in row if _is_year_token(v))
        if has_country and n_years >= min_years:
            return r
    return None

def _parse_icrg_sheet(filepath, sheet_name, key):
    """
    Read one ICRG sheet, find the Country+year header, and return
    a long DataFrame with columns: Country, Year, <key>.
    """
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name,
                           header=None, engine=ENGINE)
    hdr_row = _find_header_row(df_raw)
    if hdr_row is None:
        print(f"    WARNING: Could not find header row in sheet '{sheet_name}'")
        return None

    # Use the header row as column names
    df = df_raw.iloc[hdr_row:].copy()
    df.columns = [str(v).strip() for v in df.iloc[0]]
    df = df.iloc[1:].reset_index(drop=True)

    # Identify Country column (case-insensitive)
    country_col = next((c for c in df.columns if c.lower() == "country"), None)
    if country_col is None:
        print(f"    WARNING: No 'Country' column in sheet '{sheet_name}'")
        return None

    # Identify year columns
    year_cols = [c for c in df.columns if _is_year_token(c)]
    if not year_cols:
        print(f"    WARNING: No year columns found in sheet '{sheet_name}'")
        return None

    # Keep only Country + year columns, rename Country
    df = df[[country_col] + year_cols].rename(columns={country_col: "Country"})

    # Normalize year column names to plain integers
    rename_map = {c: int(float(c)) for c in year_cols}
    df = df.rename(columns=rename_map)
    year_int_cols = list(rename_map.values())

    # Melt to long format
    long_df = df.melt(id_vars="Country", value_vars=year_int_cols,
                      var_name="Year", value_name=key)
    long_df["Country"] = long_df["Country"].astype(str).str.strip()
    long_df["Year"]    = pd.to_numeric(long_df["Year"], errors="coerce")
    long_df[key]       = pd.to_numeric(long_df[key], errors="coerce")
    long_df = long_df.dropna(subset=["Country", "Year", key])
    long_df = long_df[long_df["Country"].str.lower() != "country"]  # drop repeated headers
    long_df = long_df[long_df["Country"].str.lower() != "nan"]
    long_df["Year"] = long_df["Year"].astype(int)
    return long_df

# Open workbook and confirm sheets
xls = pd.ExcelFile(ICRG_FILE, engine=ENGINE)
print(f"  Sheets: {xls.sheet_names}")
print()

# Parse each of the five GADP component sheets
indicator_long = {}
for sheet_name, (key, max_score) in GADP_SHEETS.items():
    if sheet_name not in xls.sheet_names:
        print(f"  WARNING: Sheet '{sheet_name}' not found — skipping")
        continue
    long_df = _parse_icrg_sheet(ICRG_FILE, sheet_name, key)
    if long_df is not None:
        indicator_long[key] = (long_df, max_score)
        print(f"  Parsed '{sheet_name}': {len(long_df)} country-year rows")

if len(indicator_long) < 2:
    raise RuntimeError("Parsed fewer than two GADP components. Check file and sheet names.")

# Merge all five components on Country + Year
dfs = [df for df, _ in indicator_long.values()]
merged = reduce(lambda a, b: pd.merge(a, b, on=["Country", "Year"], how="outer"), dfs)

# Filter to governance window
merged = merged[merged["Year"].isin(ICRG_GOV_YEARS)].copy()
print(f"\n  Rows in governance window {ICRG_GOV_YEARS[0]}-{ICRG_GOV_YEARS[-1]}: {len(merged)}")

# Normalise each component to [0, 1]
for key, (_, max_score) in indicator_long.items():
    merged[f"{key}_norm"] = merged[key] / max_score

# GADP = average of five normalised components (require all five present)
norm_cols = [f"{k}_norm" for k in indicator_long]
merged["GADP"] = merged[norm_cols].mean(axis=1, skipna=False)

# Map country names to ISO3 using ICRG_NAME_TO_ISO crosswalk
ICRG_ALIASES = {
    "Congo, DR":           "COD",
    "Trinidad & Tobago":   "TTO",
    "Korea, DPR":          "PRK",
    "West Germany":        "DEU",
    "East Germany":        "DDR",
    "Serbia & Montenegro": "SCG",
}
def _icrg_to_iso(name):
    name = str(name).strip()
    if name in ICRG_ALIASES:
        return ICRG_ALIASES[name]
    return ICRG_NAME_TO_ISO.get(name)

merged["ISO3"] = merged["Country"].map(_icrg_to_iso)

# Average GADP per country over the governance window
gadp_index = (merged.dropna(subset=["GADP", "ISO3"])
                     .groupby("ISO3", as_index=False)["GADP"]
                     .mean())

print(f"  GADP index computed for: {gadp_index.shape[0]} countries")

# Merge into master
for iso, d in master.items():
    row = gadp_index[gadp_index["ISO3"] == iso]
    d["governance"] = float(row["GADP"].iloc[0]) if not row.empty else None

matched_gov = sum(1 for d in master.values() if d.get("governance") is not None)
print(f"  Matched to master: {matched_gov} countries")

print("\n  Sample GADP values:")
for check in ["USA", "CHE", "DEU", "GBR", "NER", "NGA", "CHN"]:
    gov = master.get(check, {}).get("governance")
    print(f"    {check}: {gov:.4f}" if gov is not None else f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 7: Openness — branches on VERSION
# =============================================================================
print(f"\n>>> STEP 7: Computing openness measure (VERSION {VERSION})...")

# -----------------------------------------------------------------------------
# VERSION 1: Sachs-Warner fraction of years open (1950-1992)
# open.csv uses IDENTICAL country names to PWT 5.6 — reuse PWT56_NAME_TO_ISO
# -----------------------------------------------------------------------------
if VERSION == 1:
    open_counts = defaultdict(lambda: {"open": 0, "total": 0})
    with open(OPEN_FILE, "r", encoding="latin1") as f:
        reader = csv.DictReader(f)
        for row in reader:
            country = (row.get("COUNTRY") or "").strip()
            iso = PWT56_NAME_TO_ISO.get(country)   # reuse exact same crosswalk
            if not iso:
                continue
            open_val = str(row.get("OPEN") or "").strip()
            if open_val in ("0.00", "1.00", "0", "1"):
                open_counts[iso]["total"] += 1
                if open_val in ("1.00", "1"):
                    open_counts[iso]["open"] += 1

    openness_data = {}
    for iso, counts in open_counts.items():
        if counts["total"] > 0:
            openness_data[iso] = counts["open"] / counts["total"]

    print(f"  Sachs-Warner fractions computed for: {len(openness_data)} countries")
    print(f"  Note: Data covers 1950-1992 (H&J use 1950-1994 — minor deviation)")

    # Quick checks
    for check in ["USA", "GBR", "DEU", "CHN", "NER"]:
        v = openness_data.get(check)
        print(f"    {check}: {v:.3f}" if v is not None else f"    {check}: NA")

# -----------------------------------------------------------------------------
# VERSION 3: Fraser Institute Area 5 averaged 2015-2019
# Panel dataset sheet (sheet index 1): ISO_Code, Year, Area 5
# Area 5 is on 0-10 scale — normalise to [0,1] by dividing by 10
# -----------------------------------------------------------------------------
elif VERSION == 3:
    FRASER_YEARS = list(range(2015, 2020))
    fraser_by_country = defaultdict(list)

    _, fraser_rows = read_xlsx_sheet(FRASER_FILE, sheet_index=1)
    print(f"  Fraser Institute rows loaded: {len(fraser_rows)}")

    for row in fraser_rows:
        iso   = (row.get("ISO_Code") or "").strip()
        yr    = str(row.get("Year") or "").strip()
        area5 = safe_float(row.get("Area 5"))
        if iso and area5 is not None:
            try:
                if int(float(yr)) in FRASER_YEARS:
                    fraser_by_country[iso].append(area5)
            except:
                continue

    openness_data = {}
    for iso, vals in fraser_by_country.items():
        if vals:
            openness_data[iso] = (sum(vals) / len(vals)) / 10.0

    print(f"  Fraser Area 5 averaged for: {len(openness_data)} countries")
    print(f"  Averaging window: {FRASER_YEARS[0]}-{FRASER_YEARS[-1]}")
    print(f"  Note: Normalised from 0-10 to 0-1 scale")

# Merge openness into master
for iso, d in master.items():
    d["openness"] = openness_data.get(iso)

matched_open = sum(1 for d in master.values() if d.get("openness") is not None)
print(f"\n  Matched to master: {matched_open} countries")

print(f"\n  Sample openness values:")
for check in ["USA", "GBR", "DEU", "CHE", "NER", "CHN", "KEN"]:
    opn = master.get(check, {}).get("openness")
    print(f"    {check}: {opn:.4f}" if opn is not None else f"    {check}: NA")


In [ ]:
# =============================================================================
# STEP 8: Instruments (identical across versions)
# =============================================================================
print("\n>>> STEP 8: Merging instruments...")

# Distance from equator
geo_data = {}
with open(GEO_FILE, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        iso  = (row.get('iso3') or '').strip()
        dist = safe_float(row.get('distancefromeq'))
        if iso:
            geo_data[iso] = dist

# Frankel-Romer predicted trade share
fr_data = {}
with open(FR_FILE, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        iso   = (row.get('iso3') or '').strip()
        fr_val = safe_float(row.get('fr_constructed_trade_share'))
        if iso:
            fr_data[iso] = fr_val

# Language fractions
lang_data = {}
with open(LANG_FILE, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        iso = (row.get('iso3') or '').strip()
        eng = safe_float(row.get('english_frac'))
        we  = safe_float(row.get('we_lang_frac'))
        if iso:
            lang_data[iso] = {'english_frac': eng, 'we_lang_frac': we}

# Merge into master
for iso, d in master.items():
    d['distancefromeq'] = geo_data.get(iso)
    d['fr_trade']       = fr_data.get(iso)
    d['english_frac']   = lang_data.get(iso, {}).get('english_frac')
    d['we_lang_frac']   = lang_data.get(iso, {}).get('we_lang_frac')

print(f"  Distance from equator matched : {sum(1 for d in master.values() if d.get('distancefromeq') is not None)}")
print(f"  FR trade matched              : {sum(1 for d in master.values() if d.get('fr_trade') is not None)}")
print(f"  Language data matched         : {sum(1 for d in master.values() if d.get('english_frac') is not None)}")


In [ ]:
# =============================================================================
# STEP 9: Social Infrastructure Index
# =============================================================================
print("\n>>> STEP 9: Constructing social infrastructure index...")

# S = (governance + openness) / 2
# Both components already on [0,1] scale:
#   governance = GADP (normalised components, averaged over window)
#   openness   = Sachs-Warner fraction (V1) or Fraser Area5/10 (V3)

for iso, d in master.items():
    gov = d.get('governance')
    opn = d.get('openness')
    if gov is not None and opn is not None:
        d['social_infra'] = (gov + opn) / 2.0
    else:
        d['social_infra'] = None

n_si = sum(1 for d in master.values() if d.get('social_infra') is not None)
print(f"  Social infrastructure index computed for: {n_si} countries")

print(f"\n  Sample S values:")
for check in ['USA', 'CHE', 'NER', 'NGA', 'CHN', 'KEN']:
    si = master.get(check, {}).get('social_infra')
    print(f"    {check}: S = {si:.4f}" if si is not None else f"    {check}: S = NA")


In [ ]:
# =============================================================================
# STEP 10: Final Assembly and Quality Check
# =============================================================================
print("\n>>> STEP 10: Final assembly and quality check...")

CORE_VARS = ['yl', 'ky_ratio', 'hc', 'social_infra']

complete = {
    iso: d for iso, d in master.items()
    if all(d.get(v) is not None for v in CORE_VARS)
}

print(f"  Total countries in master        : {len(master)}")
print(f"  Complete cases (Y/L+K/Y+h+S)    : {len(complete)}")

print("\n  Missing data summary:")
check_vars = ['yl', 'ky_ratio', 'hc', 'yr_sch', 'mining_va',
              'governance', 'openness', 'social_infra',
              'distancefromeq', 'fr_trade']
for var in check_vars:
    n_miss = sum(1 for d in master.values() if d.get(var) is None)
    print(f"    {var:<20}: {n_miss:>3} missing out of {len(master)}")

# Verification table — key countries relative to USA
print(f"\n  Verification table (relative to USA):")
usa = master.get('USA', {})
usa_yl = usa.get('yl', 1)
usa_ky = usa.get('ky_ratio', 1)
usa_hc = usa.get('hc', 1)

print(f"  {'ISO':<6} {'Y/L (rel)':>10} {'K/Y (rel)':>10} {'h (rel)':>10} {'S':>8} {'yr_sch':>8}")
print(f"  {'-'*56}")
for iso in ['USA', 'CHE', 'JPN', 'GBR', 'DEU', 'FRA', 'BRA', 'CHN', 'IND', 'KEN', 'NGA', 'NER']:
    d   = master.get(iso, {})
    yl  = d.get('yl')
    ky  = d.get('ky_ratio')
    hc  = d.get('hc')
    si  = d.get('social_infra')
    sch = d.get('yr_sch')
    r_yl = f"{yl/usa_yl:.3f}"  if (yl and usa_yl) else 'NA'
    r_ky = f"{ky/usa_ky:.3f}"  if (ky and usa_ky) else 'NA'
    r_hc = f"{hc/usa_hc:.3f}"  if (hc and usa_hc) else 'NA'
    si_s = f"{si:.3f}"          if si is not None   else 'NA'
    sc_s = f"{sch:.1f}"         if sch is not None  else 'NA'
    print(f"  {iso:<6} {r_yl:>10} {r_ky:>10} {r_hc:>10} {si_s:>8} {sc_s:>8}")

# Key ratio check
ner_yl = master.get('NER', {}).get('yl')
if ner_yl and usa_yl:
    print(f"\n  USA/Niger Y/L ratio: {usa_yl/ner_yl:.1f}x  (Hall & Jones original: 35x)")


In [ ]:
# =============================================================================
# STEP 11: Save Output
# =============================================================================
print(f"\n>>> STEP 11: Saving to {OUTPUT_FILE}...")

OUTPUT_VARS = [
    'iso3', 'country',
    'yl', 'K', 'ky_ratio',
    'hc', 'hc_bl', 'yr_sch',
    'mining_va',
    'governance', 'openness', 'social_infra',
    'distancefromeq', 'fr_trade', 'english_frac', 'we_lang_frac',
]

# Add version-specific auxiliary columns
if VERSION == 1:
    OUTPUT_VARS += ['kapw_nonres']
elif VERSION == 3:
    OUTPUT_VARS += ['rgdpo', 'emp', 'csh_i', 'delta', 'labsh']

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_VARS, extrasaction='ignore')
    writer.writeheader()
    for iso in sorted(master.keys()):
        row = {var: master[iso].get(var, '') for var in OUTPUT_VARS}
        for var in OUTPUT_VARS:
            val = row[var]
            if isinstance(val, float):
                row[var] = round(val, 6)
        writer.writerow(row)

print(f"  Saved {len(master)} countries to {OUTPUT_FILE}")
print(f"\n>>> PIPELINE COMPLETE.")
print(f"    VERSION {VERSION} dataset ready for analysis.")
print(f"    Next step: Run hj_analysis.py on {OUTPUT_FILE}")
